# Annual TorNet manifest generation

Build, validate, and persist one or more annual TorNet manifests.

The initial run is intentionally limited to 2014. After 2014 is
validated, the same notebook will be extended to 2015–2022.


In [1]:
%pip install -q xarray netCDF4 pandas pyarrow


In [2]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%pip install --force-reinstall --no-deps "/content/drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.0-py3-none-any.whl"


Processing ./drive/MyDrive/TorNet_Backup/packages/tornet_detection-0.1.0-py3-none-any.whl
  Attempting uninstall: tornet-detection
    Found existing installation: tornet-detection 0.1.0
    Uninstalling tornet-detection-0.1.0:
      Successfully uninstalled tornet-detection-0.1.0


In [4]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

import tornado_detection

from tornado_detection.data.manifest import (
    EXPECTED_CATEGORIES,
    EXPECTED_SPLITS,
    build_archive_manifests,
    validate_manifests,
    write_manifest_artifacts,
)

print(
    "tornado_detection package version:",
    tornado_detection.__version__,
)
print(
    "Loaded tornado_detection from:",
    tornado_detection.__file__,
)


tornado_detection package version: 0.1.0
Loaded tornado_detection from: /usr/local/lib/python3.12/dist-packages/tornado_detection/__init__.py


## Configuration


In [5]:
TORNET_ARCHIVE_DIR = Path(
    "/content/drive/MyDrive/TorNet_Backup"
)

MANIFEST_OUTPUT_ROOT = (
    TORNET_ARCHIVE_DIR
    / "manifests"
    / "v1"
)

MANIFEST_WORK_DIR = Path(
    "/content/tornet_manifest_work"
)

YEARS_TO_BUILD = (2014,)

EXPECTED_COMBINATIONS = {
    (split, category)
    for split in EXPECTED_SPLITS
    for category in EXPECTED_CATEGORIES
}

EXPECTED_DIMENSIONS = {
    "time": 4,
    "sweep": 2,
    "azimuth": 120,
    "range": 240,
    "lims": 2,
}

print("Years to build:", YEARS_TO_BUILD)
print("Archive directory:", TORNET_ARCHIVE_DIR)
print("Output root:", MANIFEST_OUTPUT_ROOT)


Years to build: (2014,)
Archive directory: /content/drive/MyDrive/TorNet_Backup
Output root: /content/drive/MyDrive/TorNet_Backup/manifests/v1


## Build and validate one year


In [6]:
def build_and_write_year(year: int):
    archive_path = (
        TORNET_ARCHIVE_DIR
        / f"tornet_{year}.tar.gz"
    )

    output_directory = (
        MANIFEST_OUTPUT_ROOT
        / str(year)
    )

    if not archive_path.is_file():
        raise FileNotFoundError(archive_path)

    if (output_directory / "_SUCCESS.json").exists():
        raise FileExistsError(
            "A completed manifest already exists for "
            f"{year}: {output_directory}"
        )

    print()
    print("=" * 72)
    print(f"Building TorNet manifest for {year}")
    print("Archive:", archive_path)
    print("Output:", output_directory)
    print("=" * 72)

    result = build_archive_manifests(
        archive_path,
        expected_year=year,
        working_directory=MANIFEST_WORK_DIR,
        progress_every=1_000,
        progress=print,
    )

    actual_combinations = {
        (row.split, row.category)
        for row in (
            result.file_manifest[
                ["split", "category"]
            ]
            .drop_duplicates()
            .itertuples(index=False)
        )
    }

    if actual_combinations != EXPECTED_COMBINATIONS:
        raise AssertionError(
            f"Unexpected split/category combinations for {year}: "
            f"actual={sorted(actual_combinations)}, "
            f"expected={sorted(EXPECTED_COMBINATIONS)}"
        )

    if len(result.file_manifest) != result.netcdf_member_count:
        raise AssertionError(
            "File-manifest count does not match scanned NetCDF "
            f"members for {year}: "
            f"files={len(result.file_manifest):,}, "
            f"members={result.netcdf_member_count:,}"
        )

    validation = validate_manifests(
        result,
        expected_file_count=result.netcdf_member_count,
        expected_frame_count=(
            result.netcdf_member_count * 4
        ),
        expected_frames_per_file=4,
        expected_dimensions=EXPECTED_DIMENSIONS,
    )

    display(validation.checks)
    display(validation.category_frame_summary)
    display(result.schema_summary)

    print("Event groups crossing official splits:")
    display(validation.event_split_overlap)

    print(
        "Episode groups crossing official splits "
        "(informational):"
    )
    display(validation.episode_split_overlap)

    validation.assert_valid()

    artifacts = write_manifest_artifacts(
        result,
        validation,
        output_directory,
        overwrite=False,
    )

    summary = {
        "year": year,
        "netcdf_members": result.netcdf_member_count,
        "file_rows": len(result.file_manifest),
        "frame_rows": len(result.frame_manifest),
        "schema_variants": len(result.schema_summary),
        "build_errors": len(result.errors),
        "event_split_overlaps": len(
            validation.event_split_overlap
        ),
        "episode_split_overlaps": len(
            validation.episode_split_overlap
        ),
        "positive_frames": int(
            result.frame_manifest[
                "frame_label"
            ].sum()
        ),
        "total_frames": len(result.frame_manifest),
        "output_directory": str(output_directory),
    }

    print()
    print(
        f"PASS: {year} manifest built, validated, "
        "and written"
    )

    for artifact_name, artifact_path in artifacts.items():
        print(f"- {artifact_name}: {artifact_path}")

    return result, validation, summary


## Run the configured years


In [7]:
annual_results = {}
annual_validations = {}
annual_summaries = []

for year in YEARS_TO_BUILD:
    result, validation, summary = (
        build_and_write_year(year)
    )

    annual_results[year] = result
    annual_validations[year] = validation
    annual_summaries.append(summary)

annual_summary_df = pd.DataFrame(
    annual_summaries
)

display(annual_summary_df)

print(
    "PASS: all configured annual manifest builds "
    "completed"
)



Building TorNet manifest for 2014
Archive: /content/drive/MyDrive/TorNet_Backup/tornet_2014.tar.gz
Output: /content/drive/MyDrive/TorNet_Backup/manifests/v1/2014
Scanned 1,000 NetCDF members; built 1,000 file rows and 4,000 frame rows; errors=0
Scanned 2,000 NetCDF members; built 2,000 file rows and 8,000 frame rows; errors=0
Scanned 3,000 NetCDF members; built 3,000 file rows and 12,000 frame rows; errors=0
Scanned 4,000 NetCDF members; built 4,000 file rows and 16,000 frame rows; errors=0
Scanned 5,000 NetCDF members; built 5,000 file rows and 20,000 frame rows; errors=0
Scanned 6,000 NetCDF members; built 6,000 file rows and 24,000 frame rows; errors=0
Scanned 7,000 NetCDF members; built 7,000 file rows and 28,000 frame rows; errors=0
Scanned 8,000 NetCDF members; built 8,000 file rows and 32,000 frame rows; errors=0
Scanned 9,000 NetCDF members; built 9,000 file rows and 36,000 frame rows; errors=0
Scanned 10,000 NetCDF members; built 10,000 file rows and 40,000 frame rows; errors

,check,required,passed,observed,expected,detail
0,build_errors,True,True,0,0,
1,file_row_count,True,True,19993,19993,
2,frame_row_count,True,True,79972,79972,
3,unique_archive_members,True,True,19993,19993,
4,unique_file_ids,True,True,19993,19993,
5,unique_frame_ids,True,True,79972,79972,
6,frame_rows_match_file_frame_counts,True,True,0,0,
7,frame_label_sums_match_file_manifest,True,True,0,0,
8,frame_indices_are_contiguous,True,True,0,0,
9,expected_frames_per_file,True,True,0,0,Expected 4 frames for every file


,split,category,file_count,frame_count,positive_frame_count,files_with_positive_frames,files_with_mixed_frame_labels,positive_frame_prevalence
0,test,NUL,1345,5380,0,0,0,0.000000
1,test,TOR,353,1412,792,353,278,0.560907
2,test,WRN,848,3392,0,0,0,0.000000
3,train,NUL,9934,39736,0,0,0,0.000000
4,train,TOR,1176,4704,2260,1176,1036,0.480442
5,train,WRN,6337,25348,0,0,0,0.000000


,schema_fingerprint,file_count,splits_json,categories_json,first_archive_member,schema_payload_json
0,cc3cc0a1a9b6023a6ef63d80d5fc1bf28287a687930d06...,12808,"[""test"",""train""]","[""NUL"",""TOR""]",train/2014/NUL_140430_100450_KRAX_1073853n_D0.nc,"{""coordinates"":[""azimuth"",""range"",""time""],""dat..."
1,b9f3dbdfdbb44d5e179118408883b6f64bf6e4ce0fa96f...,7185,"[""test"",""train""]","[""WRN""]",train/2014/WRN_140714_222212_KLWX_1074462n_C4.nc,"{""coordinates"":[""azimuth"",""range"",""time""],""dat..."


Event groups crossing official splits:


,event_group_id,splits_json,file_count,archive_members_json
0,505771,"[""test"",""train""]",4,"[""test/2014/WRN_140430_004059_KRAX_1073839n_L0..."


Episode groups crossing official splits (informational):


,episode_id,splits_json,file_count,archive_members_json
0,83781,"[""test"",""train""]",22,"[""test/2014/WRN_140430_004059_KRAX_1073839n_L0..."
1,83782,"[""test"",""train""]",117,"[""test/2014/TOR_140429_013947_KBMX_508054_K7.n..."
2,84594,"[""test"",""train""]",89,"[""test/2014/TOR_140606_193553_KLZK_510878_K7.n..."
3,85260,"[""test"",""train""]",72,"[""test/2014/WRN_140429_141343_KTLH_1073826n_Y5..."
4,88050,"[""test"",""train""]",30,"[""test/2014/WRN_140824_232808_KDLH_1074569n_H7..."
5,90774,"[""test"",""train""]",19,"[""test/2014/TOR_141006_234057_KHTX_544825_N2.n..."
6,90857,"[""test"",""train""]",32,"[""test/2014/WRN_141006_211340_KJKL_1074657n_V8..."


AssertionError: Required TorNet manifest validations failed:
- event_groups_do_not_cross_official_splits: observed=1; expected=0